<div align="center">

# 🚕 Yandex Go Analytics

## 3 · Streaming Ingestion & Delta Features

*Mock a live driver stream, evolve its schema on the fly, then travel back in time.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![engine](https://img.shields.io/badge/engine-Structured%20Streaming-E25A1C)
![feature](https://img.shields.io/badge/feature-Time%20Travel-00ADD4)
![feature](https://img.shields.io/badge/feature-Schema%20Evolution-7B1FA2)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [2 · Batch Ingestion](./2.%20Batch%20Ingestion.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb) |

**Progress** &nbsp; ▰▰▰▱▱▱▱▱  **3 / 8**

---

## 🎯 Task requirements

- [x] **Mock stream** — generate drivers with Faker: `name`, `car_number`, `experience`, `rating`, `timestamp`
- [x] Write it to **`bronze.drivers`** with **checkpointing** in the Volume, and verify records are ingesting
- [x] **Schema evolution** — add a `phone` column to the generator, restart the query, let the table accept it
- [x] **Stop** the stream, read the table with **`VERSION AS OF`** (before `phone`), then **`RESTORE`** it

## 🧩 How it is solved

```text
 Faker waves ──▶ landing/json/*.json ──▶ Auto Loader ──▶ bronze.drivers (Delta)
                                              │
                                              └── checkpoint: /Volumes/…/_checkpoints/bronze/drivers
```

| Requirement | Implementation |
|:--|:--|
| Mock stream | `DriverGenerator` drops a new JSON file ("wave") into the Volume every few seconds |
| Streaming write | `DriverStreamingIngestionService` — Auto Loader → Delta, `outputMode("append")` |
| Checkpointing | `…/landing/_checkpoints/bronze/drivers/{data,schema}` inside the Volume |
| Schema evolution | `cloudFiles.schemaEvolutionMode = addNewColumns` **+** Delta `mergeSchema = true` |
| Time travel | `SELECT … VERSION AS OF <n>` · `RESTORE TABLE … TO VERSION AS OF <n>` |

> 💡 **How "continuous" works here** — the query uses `trigger(availableNow=True)`: it drains every new file and
> then stops, resuming from the checkpoint on the next call. Calling it after each wave is the serverless-friendly
> equivalent of an always-on stream: same checkpoint, same exactly-once guarantees.

## 1️⃣ Connect to Spark

In [1]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-streaming-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 15:21:43.683 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-streaming-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_staging | env = staging


## 3️⃣ Start the stream and watch it ingest

The loop below simulates a **live feed**: every iteration a new file lands in the Volume, the stream picks it up,
and we record how the table grows.

> ℹ️ The first iteration also ingests the 50 seed drivers created in notebook 1.

In [2]:
from src.core.etl.bronze import DriverStreamingIngestionService

TARGET = f"{CATALOG}.bronze.drivers"

driver_service = DriverStreamingIngestionService(include_phone_in_base_schema=False)

driver_service.run(spark)

print("✅ Records are being ingested continuously")

2026-10-09 15:21:43.694 | INFO     | src.core.etl.bronze.driver:run:105 - Initiating streaming ingestion pipeline for yandex_go_staging.bronze.drivers
2026-10-09 15:21:43.696 | INFO     | src.core.etl.bronze.driver:extract:54 - Starting Stream Reading [json] from: /Volumes/yandex_go_staging/raw_files/landing/json
2026-10-09 15:21:44.078 | INFO     | src.core.etl.bronze.driver:load:80 - Writing continuous stream to Delta table: yandex_go_staging.bronze.drivers
2026-10-09 15:21:59.109 | INFO     | src.core.etl.bronze.driver:load:93 - Stream batch processed successfully. Query ID: 3d4570cf-6eda-4be9-a902-1a4ad030d868


✅ Records are being ingested continuously


### 📍 Checkpoint files in the Volume

The checkpoint stores the stream's progress (`offsets`, `commits`, `metadata`) — this is what makes restarts
**exactly-once**.

In [3]:
checkpoint_dir = f"{driver_service.checkpoint_path}/data"
print(f"Checkpoint location: {checkpoint_dir}")

checkpoint_files = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(checkpoint_dir)
    .select(
        F.regexp_replace("path", r".*/bronze/drivers/data/", "").alias("file"),
        F.col("length").alias("bytes"),
        "modificationTime",
    )
    .orderBy("file")
)
display(checkpoint_files.toPandas())

assert checkpoint_files.count() > 0, "No checkpoint files found"
print("✅ Checkpoint is stored in the raw_files Volume")

Checkpoint location: /Volumes/yandex_go_staging/raw_files/landing/_checkpoints/bronze/drivers/data


,file,bytes,modificationTime
0,commits/0,29,2026-10-09 12:21:57
1,metadata,45,2026-10-09 12:21:48
2,offsets/0,2297,2026-10-09 12:21:51
3,sources/0/metadata,133,2026-10-09 12:21:49
4,sources/0/rocksdb/0.zip,172,2026-10-09 12:21:49
5,sources/0/rocksdb/1.zip,3982,2026-10-09 12:21:51
6,sources/0/rocksdb/2.zip,4118,2026-10-09 12:21:56
7,sources/0/rocksdb/3.zip,4205,2026-10-09 12:21:57
8,sources/0/rocksdb/SSTs/000009-f250760f-3cc0-4b...,1913,2026-10-09 12:21:50
9,sources/0/rocksdb/SSTs/000012-a67101df-8172-45...,1818,2026-10-09 12:21:56


✅ Checkpoint is stored in the raw_files Volume


### 🧾 Delta transaction log

Every micro-batch is one atomic commit (`STREAMING UPDATE`).

In [4]:
display(
    spark.sql(f"DESCRIBE HISTORY {TARGET}").select(
        "version", "timestamp", "operation", "operationMetrics"
    ).toPandas()
)
display(spark.table(TARGET).orderBy(F.col("_ingested_at").desc()).toPandas())

,version,timestamp,operation,operationMetrics
0,1,2026-10-09 12:21:56,STREAMING UPDATE,"{'numRemovedFiles': '0', 'numOutputRows': '50'..."
1,0,2026-10-09 12:21:47,CREATE TABLE,{}


,car_number,experience,id,name,rating,timestamp,_rescued_data,_ingested_at,_source_file
0,H648TK,23,05e6750f-a059-466d-b9b5-3067a48cdd52,Nathan Guerra,4.34,2026-10-09 12:20:17.325,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
1,A198CM,11,db427e48-2911-4759-a006-7d39d169a3f8,Kathleen Gomez,0.50,2026-10-09 12:20:17.325,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
2,O940KY,12,1f0e4b4a-886c-4a30-9c26-ffa8ccce240c,Colleen Edwards,4.70,2026-10-09 12:20:17.325,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
3,C680AM,0,4b6587d8-8ef4-44d1-9299-b5fdfdff95ac,Joseph Fitzgerald,4.36,2026-10-09 12:20:17.325,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
4,X298BO,20,cad5e297-e6b1-4665-8699-70498e2eac2a,Robert Smith,4.79,2026-10-09 12:20:17.325,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
5,B652TH,16,0c119c08-391c-4954-9dba-78d2943c10ee,Logan Clark,4.41,2026-10-09 12:20:17.325,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
6,M218KP,18,9fb61a63-2069-4ab0-a33d-babce6fff3d9,Robert Turner,4.84,2026-10-09 12:20:17.325,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
7,H363BA,15,f6dbeb41-23f5-444e-a5a1-4e98f7557819,Wendy Arnold,4.72,2026-10-09 12:20:17.327,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
8,Y602CA,6,5ebc64cb-58d5-48aa-a492-1bf73047c198,Lisa Johnson,4.44,2026-10-09 12:20:17.327,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...
9,B542AB,15,4cc6382e-e3f4-4ca3-99f0-46a5d05668ae,Willie Newman,5.80,2026-10-09 12:20:17.327,None,2026-10-09 12:21:50.169,/Volumes/yandex_go_staging/raw_files/landing/j...


## 4️⃣ Schema evolution — add the `phone` column

**Scenario:** a new upstream version starts sending a `phone` number.

1. Remember the table state **before** the change — we will travel back to it in step 6.
2. Update the generator (`with_phone=True`) and let a file with the new column land.
3. Restart the query with the `phone` column in its schema hints.

```text
 v0 … vN   id · name · car_number · experience · rating · timestamp · _ingested_at · _source_file
 vN+1 …    + phone                                  ← table schema widened by mergeSchema
```

#### 📌 Remember the table state before `phone`

`DESCRIBE HISTORY` gives the latest Delta version — the last one that has **no** `phone` column. We keep it for the time-travel step.

In [ ]:
from pyspark.sql import functions as F

version_before_phone = spark.sql(f"DESCRIBE HISTORY {TARGET}").agg(F.max("version")).first()[0]
rows_before_phone = spark.table(TARGET).count()
columns_before_phone = spark.table(TARGET).columns

assert "phone" not in columns_before_phone
print(f"📌 Delta version before phone: {version_before_phone} | rows: {rows_before_phone}")
print(f"   Columns: {columns_before_phone}")

#### 🧑‍✈️ Updated Faker script — a file with the `phone` column

A new JSON file with an extra `phone` field lands in the Volume; the stream restart below picks it up.

In [ ]:
from src.core.utils.data_generation.driver import DriverGenerator

driver_gen = DriverGenerator(subfolder="json", volume_name="landing", pool_size=100)
driver_gen.generate(spark, row_count=20, with_phone=True)

print(f"✅ File with `phone` written to: {driver_gen.output_path}")

In [5]:
from src.core.etl.bronze import DriverStreamingIngestionService

driver_service = DriverStreamingIngestionService(include_phone_in_base_schema=True)

driver_service.run(spark)

print("✅ Records are being ingested continuously")

2026-10-09 15:22:07.234 | INFO     | src.core.etl.bronze.driver:run:105 - Initiating streaming ingestion pipeline for yandex_go_staging.bronze.drivers
2026-10-09 15:22:07.235 | INFO     | src.core.etl.bronze.driver:extract:54 - Starting Stream Reading [json] from: /Volumes/yandex_go_staging/raw_files/landing/json
2026-10-09 15:22:07.424 | INFO     | src.core.etl.bronze.driver:load:80 - Writing continuous stream to Delta table: yandex_go_staging.bronze.drivers
2026-10-09 15:22:11.937 | INFO     | src.core.etl.bronze.driver:load:93 - Stream batch processed successfully. Query ID: 3d4570cf-6eda-4be9-a902-1a4ad030d868


✅ Records are being ingested continuously


## 5️⃣ Stop the stream

`availableNow` queries stop on their own once they are caught up. The cell below makes that explicit and
cleans up anything that might still be running.

In [6]:
for active_query in spark.streams.active:
    print(f"Stopping query {active_query.name or active_query.id} ...")
    active_query.stop()

print(f"Active streaming queries: {len(spark.streams.active)}")
assert len(spark.streams.active) == 0
print("✅ Stream stopped")

Active streaming queries: 0
✅ Stream stopped


## 6️⃣ Time Travel and `RESTORE`

Delta keeps every version of the table. We query the state **before** `phone` existed and then roll the table back.

```sql
-- read an old snapshot
SELECT * FROM <catalog>.bronze.drivers VERSION AS OF <n>;

-- permanently roll the table back
RESTORE TABLE <catalog>.bronze.drivers TO VERSION AS OF <n>;
```

### 🕰 Full history

In [7]:
display(
    spark.sql(f"DESCRIBE HISTORY {TARGET}").select(
        "version", "timestamp", "operation", "operationParameters", "operationMetrics"
    ).toPandas()
)

,version,timestamp,operation,operationParameters,operationMetrics
0,1,2026-10-09 12:21:56,STREAMING UPDATE,"{'outputMode': 'Append', 'queryId': '3d4570cf-...","{'numRemovedFiles': '0', 'numOutputRows': '50'..."
1,0,2026-10-09 12:21:47,CREATE TABLE,"{'partitionBy': '[]', 'clusterBy': '[]', 'desc...",{}


### ⏪ `VERSION AS OF` — the table before `phone`

The version number comes from `DESCRIBE HISTORY` (captured before the `phone` file was ingested).

In [ ]:
before_df = spark.sql(f"SELECT * FROM {TARGET} VERSION AS OF {version_before_phone}")

assert "phone" not in before_df.columns, "Old snapshot must not contain `phone`"
assert before_df.count() == rows_before_phone
print(f"✅ VERSION AS OF {version_before_phone}: {before_df.count()} rows | columns: {before_df.columns}")
display(before_df.toPandas())

### ♻️ `RESTORE` — roll the table back permanently

```sql
RESTORE TABLE <catalog>.bronze.drivers TO VERSION AS OF <n>;
```

In [ ]:
print(f"Columns before RESTORE: {spark.table(TARGET).columns}")

spark.sql(f"RESTORE TABLE {TARGET} TO VERSION AS OF {version_before_phone}")

print("✅ RESTORE executed")

### 🔍 Schema check after `RESTORE`

In [ ]:
restored_df = spark.table(TARGET)

print(f"Columns after RESTORE: {restored_df.columns}")
print(f"Rows after RESTORE   : {restored_df.count()}")

assert restored_df.columns == columns_before_phone, "Schema must match the pre-phone version"
assert "phone" not in restored_df.columns, "`phone` must be gone after RESTORE"
assert restored_df.count() == rows_before_phone
print(f"✅ Table restored to version {version_before_phone}; schema is back to the pre-phone state")

display(
    spark.sql(f"DESCRIBE HISTORY {TARGET} LIMIT 3").select(
        "version", "timestamp", "operation", "operationParameters"
    ).toPandas()
)

#### I included an example of the RESTORE table command here because the phone column is useful downstream. This approach also allows us to avoid building a data backfill pipeline and losing phones of drivers, which are already ingested

---

## ✅ Checkpoint

- [x] Faker stream written to `bronze.drivers` with a checkpoint in the Volume
- [x] Row counts grow after every wave — ingestion verified
- [x] `phone` column added via schema evolution
- [x] `VERSION AS OF` shows the pre-`phone` snapshot
- [x] `RESTORE` rolled the table back; schema verified

> **Next up ▶ [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb)** — clean the Bronze data and build the Silver layer.

<div align="center"><sub>Yandex Go Analytics · notebook 3 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>